# Milestone 2 — local parsing and audited output

Prepare M1 separately with `milestone2_prepare.ipynb`. This notebook consumes a prepared raw snapshot on local disk. The input/gathering, hierarchy/sentences, bibliography/key-map and final processing modules from all four owners are integrated. Local integration checks run real owner APIs; human source/output review and full-corpus acceptance remain pending. No success rate or benchmark is implied by opening this notebook.

The chosen workflow (Danh, plan 10.0) is sequential: validate the corpus once per run; process all versions of one publication; finalize, stage/write/read-back/checkpoint; release its trees. Metrics accumulate from every publication, including blocked/empty/error. No workers before a real 50–100-publication pilot, stage profiles, output comparison and RAM evidence.


In [ ]:
from pathlib import Path
PROJECT_DIR = None
MEMBER_NAME = 'Danh'  # Danh / Khanh / Khang / Nguyen; each person selects their own folder.
DATASET_NAME = 'Danh2'
PREPARATION_DIR = Path('/content/drive/MyDrive/arXivProject/preparationM2')
PREPARED_ARCHIVE = PREPARATION_DIR / 'transport' / 'Danh2-<snapshot>.zip'
PREPARED_REMOTE_FILE_ID = ''  # remote_verified publication['archive']['id']; avoids Drive mount visibility delay.
PREPARED_SHA256 = ''          # remote_verified publication['sha256'].
PREPARED_SNAPSHOT_ID = ''     # remote_verified publication['snapshot_id'].
PREPARED_PAPERS_PENDING = 0   # remote_verified publication['pending'].
LOCAL_WORK_DIR = Path('/content/arxiv-m2-work')
RAW_CORPUS_ROOT = LOCAL_WORK_DIR / 'raw' / DATASET_NAME
OUTPUT_ROOT = LOCAL_WORK_DIR / 'parsed' / DATASET_NAME
DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/arXivProject/Milestone2/parsed') / MEMBER_NAME / DATASET_NAME
STAGE_INPUT = False
ALLOW_PARTIAL_PILOT = False  # True chỉ khi cố ý parse snapshot pilot còn pending.
RUN_PARSER = False
RUN_BENCHMARK = False
RESUME = False
STRICT = True
BACKUP_ENABLED = True
BACKUP_BATCH_SIZE = 50
RESTORE_ARCHIVES = []  # Verified parsed ZIP batches, never raw M1 backups.
REFERENCE_REVIEW_PATH = None  # Local JSON; snapshot and fingerprints must match.
CODE_REVISION = None  # None records git HEAD; actual source bytes also enter the cache hash.
RAM_SAMPLE_INTERVAL_SECONDS = 0.1
COLAB_ACCELERATOR_OBSERVED = None  # Set string 'None' ONLY after viewing Colab CPU settings.
result = None


## Open the intended checkout

On Colab, bring the reviewed checkout and install `src/requirements.txt`; Drive is mounted automatically when parsing/benchmark backup is enabled; you can also mount it explicitly with `google.colab.drive.mount('/content/drive')` when staging or backing up. Parser output stays local. Choose a dataset/snapshot covering the intended scope and all versions of each publication. A prepared batch with pending papers keeps those expected publications blocked; it does not prove full assignment coverage.


In [ ]:
import json
import os
import subprocess
import sys
if sys.version_info < (3, 11):
    raise RuntimeError('Python >=3.11 is required.')
if PROJECT_DIR is None:
    PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'src/parsing/contracts.py').is_file()), None)
if PROJECT_DIR is None:
    raise RuntimeError('Set PROJECT_DIR to the reviewed checkout.')
PROJECT_DIR = Path(PROJECT_DIR).resolve()
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from parsing.contracts import ParseConfig, CONTRACT_VERSION
from parsing.pipeline import run, run_with_backup, backup_run_batch, restore_run_batch
from parsing.benchmark import benchmark_run
from parsing.serialization import read_publication
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if BACKUP_ENABLED and (RUN_PARSER or RUN_BENCHMARK):
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
    elif DRIVE_BACKUP_ROOT.as_posix().startswith('/content/drive/'):
        raise RuntimeError('Local run: set BACKUP_ENABLED=False or select a real local backup directory.')
elif STAGE_INPUT and IN_COLAB and not PREPARED_REMOTE_FILE_ID:
    from google.colab import drive
    drive.mount('/content/drive')
print('Contract:', CONTRACT_VERSION)
print('Raw local:', RAW_CORPUS_ROOT, 'Output local:', OUTPUT_ROOT)
print('Owner/dataset:', MEMBER_NAME, DATASET_NAME, 'Backup:', BACKUP_ENABLED, DRIVE_BACKUP_ROOT)
print('Integrated owner APIs loaded. Inspect diagnostics and finish human source/output review before acceptance.')


## Stage prepared raw and optionally restore verified parsed batches

`stage_prepared_archive` verifies/copies one archive, safely extracts into a new local destination, and never restores M1 itself. The input adapter still performs full local validation at the beginning of each run, including resume. Restore parsed batches into separate fresh children of OUTPUT_ROOT; the next run verifies current input/config/code/review signatures and every artifact before accepting a cached publication. Different or damaged caches are rejected.

Raw + archive + output + staging batches must fit local disk. Local work after the last verified backup receipt may need to be rerun if the runtime disappears. A local checkpoint is not a durable Drive backup.


In [ ]:
if STAGE_INPUT:
    from preparation.transport import stage_prepared_archive
    if PREPARED_REMOTE_FILE_ID:
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        import hashlib
        from uuid import uuid4
        if not PREPARED_SHA256 or not PREPARED_SNAPSHOT_ID:
            raise ValueError('Remote archive requires SHA-256 and snapshot ID from remote_verified publication.')
        auth.authenticate_user()
        PREPARED_ARCHIVE = LOCAL_WORK_DIR / 'downloads' / (PREPARED_REMOTE_FILE_ID + '.zip')
        PREPARED_ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
        if PREPARED_ARCHIVE.exists():
            with PREPARED_ARCHIVE.open('rb') as stream: cached_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
            if cached_sha != PREPARED_SHA256: PREPARED_ARCHIVE.unlink()
        if not PREPARED_ARCHIVE.exists():
            pending = PREPARED_ARCHIVE.with_name(PREPARED_ARCHIVE.name + '.part-' + uuid4().hex)
            try:
                with pending.open('xb') as stream:
                    downloader = MediaIoBaseDownload(stream, build('drive', 'v3').files().get_media(fileId=PREPARED_REMOTE_FILE_ID))
                    done = False
                    while not done:
                        status, done = downloader.next_chunk()
                        if status: print(f'[downloading_verified] {int(status.progress() * 100)}%', flush=True)
                pending.replace(PREPARED_ARCHIVE)
            except BaseException:
                pending.unlink(missing_ok=True)
                raise
        receipt = {'kind': 'm2-prepared-transport-1.0', 'remote_verified': True, 'sha256': PREPARED_SHA256, 'snapshot_id': PREPARED_SNAPSHOT_ID, 'papers_pending': PREPARED_PAPERS_PENDING}
    else:
        receipt = json.loads(Path(str(PREPARED_ARCHIVE) + '.receipt.json').read_text(encoding='utf-8'))
    if receipt.get('kind') != 'm2-prepared-transport-1.0' or receipt.get('remote_verified') is not True:
        raise ValueError('Expected a remotely verified prepared transport receipt.')
    if receipt.get('papers_pending', 0) and not ALLOW_PARTIAL_PILOT:
        raise ValueError('Prepared transport is a partial pilot; set ALLOW_PARTIAL_PILOT=True to opt in.')
    RAW_CORPUS_ROOT = stage_prepared_archive(PREPARED_ARCHIVE, RAW_CORPUS_ROOT,
                                             expected_sha256=receipt['sha256'])
    completion = json.loads((RAW_CORPUS_ROOT / 'preparation.json').read_text(encoding='utf-8'))
    if completion.get('snapshot_id') != receipt['snapshot_id']:
        raise ValueError('Archive snapshot mismatch.')
    actual_pending = completion.get('batch', {}).get('papers_pending', 0)
    if receipt.get('papers_pending', actual_pending) != actual_pending:
        raise ValueError('Archive pending-paper count differs from receipt.')
    if actual_pending and not ALLOW_PARTIAL_PILOT:
        raise ValueError('Prepared archive is a partial pilot; set ALLOW_PARTIAL_PILOT=True to opt in.')
for archive in RESTORE_ARCHIVES:
    archive = Path(archive)
    receipt = json.loads(Path(str(archive) + '.receipt.json').read_text(encoding='utf-8'))
    restored = restore_run_batch(archive, OUTPUT_ROOT / ('restore-' + archive.stem),
                                expected_sha256=receipt['sha256'])
    print('Restored, awaiting current-run cache verification:', restored)
if RAW_CORPUS_ROOT.is_dir():
    import csv
    inventory = RAW_CORPUS_ROOT / 'inventory.csv'
    if inventory.is_file():
        with inventory.open(encoding='utf-8', newline='') as stream:
            from itertools import islice
            print('Inventory preview (not validation):', list(islice(csv.DictReader(stream), 5)))
else:
    print('No local raw selected yet; stage/select prepared input before opting in.')


## Run, audit and measure

Select MEMBER_NAME and DATASET_NAME before running. Backup defaults on and writes under Milestone2/parsed/<member>/<dataset>/<run_id>/. Keep every ZIP and matching receipt, including metadata-only final batches. The archive contains a batch manifest for restore and run_manifest.json for complete coverage/status; audits, profiles, metrics and environment are also retained, including failed publications. Keep preparationM2 separately for M3 raw inputs and M1 metadata.

Use RUN_PARSER for ordinary execution; RESUME verifies current inputs before looking for cache. BACKUP_ENABLED uses immutable batches at publication boundaries, rereads the destination hash and publishes a receipt only after verification. Backup errors preserve local good data and leave a partial run.

Use RUN_BENCHMARK separately with a **new OUTPUT_ROOT**, RESUME=False and no restore. On Colab open Runtime → Change runtime type, select CPU/accelerator None, and record `COLAB_ACCELERATOR_OBSERVED = 'None'`. Without runtime evidence the receipt stays local/unverified. Wall time includes input validation through output/read-back/checkpoints/metrics. Setup, prepare, stage and Drive backup are separate. RSS is observed process-tree peak at the declared sample interval; missing measurements stay null with errors. `benchmark/output_inventory.json` states the exact byte boundary. Fresh and cached/retry measurements must never be pooled as a fresh benchmark.


In [ ]:
if RUN_PARSER and RUN_BENCHMARK:
    raise ValueError('Choose ordinary parsing OR fresh benchmark.')
if RUN_PARSER or RUN_BENCHMARK:
    if CODE_REVISION is None:
        CODE_REVISION = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR, text=True).strip()
    config = ParseConfig(strict=STRICT, code_revision=CODE_REVISION,
                         reference_review_path=REFERENCE_REVIEW_PATH,
                         ram_sample_interval_seconds=RAM_SAMPLE_INTERVAL_SECONDS)
    if RUN_BENCHMARK:
        if RESUME or RESTORE_ARCHIVES:
            raise ValueError('Fresh benchmark cannot resume or restore parsed caches.')
        if COLAB_ACCELERATOR_OBSERVED is not None:
            os.environ['M2_COLAB_ACCELERATOR'] = COLAB_ACCELERATOR_OBSERVED
        receipt = benchmark_run(RAW_CORPUS_ROOT, OUTPUT_ROOT, config)
        print('Measured receipt:', receipt)
        print('Official CPU Colab evidence:', receipt.cpu_only_colab)
        if BACKUP_ENABLED:
            print('Backup, outside parser benchmark:', backup_run_batch(OUTPUT_ROOT / receipt.run_id, DRIVE_BACKUP_ROOT))
    elif BACKUP_ENABLED:
        result = run_with_backup(RAW_CORPUS_ROOT, OUTPUT_ROOT, config, DRIVE_BACKUP_ROOT,
                                 resume=RESUME, batch_size=BACKUP_BATCH_SIZE)
    else:
        result = run(RAW_CORPUS_ROOT, OUTPUT_ROOT, config, resume=RESUME)
    if result is not None:
        from collections import Counter
        print('Processed publications:', result.processed_publications)
        print('Diagnostic counts by code:', dict(Counter(d.code for d in result.diagnostics)))
        print('Manifest:', result.manifest_path)
        manifest = json.loads(result.manifest_path.read_text(encoding='utf-8'))
        print('Run/snapshot/status:', manifest['run_id'], manifest['snapshot_id'], manifest['status'])
        print('Metrics:', result.metrics_path)
        if result.metrics_path.is_file():
            measured = json.loads(result.metrics_path.read_text(encoding='utf-8'))
            print('Version status counts:', measured['summary']['status_counts'])
        for record in manifest['publications'][:3]:
            print('Publication audit:', {key: record[key] for key in ('arxiv_id', 'expected_versions', 'export_ready', 'working_path', 'published_path', 'audit_path', 'used_cache')})
            if record['working_path']:
                publication = read_publication(result.manifest_path.parent / record['working_path'])
                print('Read-back states/root map:', publication.version_states, publication.root_ids)
        if result.status != 'success':
            print('Not complete: inspect all diagnostics, blocked versions and failed writes.')
else:
    print('Not run. No parse result or benchmark numbers are produced by this smoke execution.')


## Acceptance and handoff (not automatically signed)

- Real multi-file publication with >=2 versions: raw → tree → normalized tree → three tables and unified bib. Compare citations against bibtex; inspect math, table cells and captions for meaning.
- Unchanged sentence moved to another parent: one element and all edges; repeated occurrences retain order. Changed paragraph body/reversed items produce different component IDs.
- Open JSON then read back: authors lists, null year/URL, tuple-key map rows, integer version/root maps and final statuses. All-no_tex is covered without fake roots; blocked/partial never become success.
- Independently alter source/config/code/review and rerun; stale cache must not be used. Interrupt/fail writes in fixtures; previous good output remains available.
- Pilot/profile 50–100 diverse real papers, compare all tables/occurrences/cites/status/coverage and p50/p95/RAM before considering workers. Stage timings are in profile/*.json; no speed promise without measurement.
- On fresh CPU Colab, calculate wall/mean/output bytes for one sample from raw receipts; look at CPU settings. Local smoke is not Colab evidence.
- Book acceptance needs metrics from the intended real run and human review: clean execute/build, HTML/Pages search/status filter/empty/reset, runtime hover/zoom/reset/run filter. Store build log, URL, commit and human checklist outside code Git.
- Khánh signs coverage/pipeline; Khang signs normalization/tree; Nguyên signs bib/cites. Do not fill reviewer names as approvals or claim M2 complete while these checks are pending. Handoff follows assignment section 12.
